## 0. Weights & Biases 설정

In [1]:
import os
import wandb

# 이미 로그인되어 있다면 저장된 인증 정보를 재사용합니다.
wandb.login()
os.environ["WANDB_PROJECT"] = "InterBook"
os.environ["WANDB_LOG_MODEL"] = "false"
RUN_NAME = "llama-3.2-1b-multi-qa-lora-3epoch-1000steps-mps32gb"
print("W&B 프로젝트:", os.environ["WANDB_PROJECT"])
print("W&B 실행 이름:", RUN_NAME)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/dongho/.netrc.
wandb: Currently logged in as: 1112hoya (Globals) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


W&B 프로젝트: InterBook
W&B 실행 이름: llama-3.2-1b-multi-qa-lora-3epoch-1000steps-mps32gb


## 1.라이브러리 및 학습 환경

In [2]:
from pathlib import Path
import json, math, random, unicodedata
from dataclasses import dataclass
import torch
import psutil
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from transformers import TrainerCallback


MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
SEED = 42
EPOCHS = 3
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM = 4
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
random.seed(SEED)
torch.manual_seed(SEED)

SAVE_ROOT = Path.home() / "InterBook" / "models" / "multi_qa_wandb_1000steps_mps32gb"
OUTPUT_DIR = str(SAVE_ROOT / "checkpoints")
ADAPTER_DIR = str(SAVE_ROOT / "adapter")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("장치:", DEVICE)
print("결과 저장:", SAVE_ROOT)
print("PyTorch:", torch.__version__)

/Users/dongho/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 19:04:07.091000 13579 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


장치: mps
결과 저장: /Users/dongho/InterBook/models/multi_qa_wandb_1000steps_mps32gb
PyTorch: 2.14.1


## 2. 데이터 경로 설정

In [3]:
BASE_DIR = Path("/Volumes/Samsung_T5/책_AI/013.동화 이해도 테스트를 위한 질의응답쌍 생성 데이터/3.개방데이터/1.데이터")

def resolve_unicode_path(path):
    path = Path(path)
    if path.exists():
        return path
    current = Path(path.anchor)
    for part in path.parts[1:]:
        target = unicodedata.normalize("NFC", part)
        match = next((p for p in current.iterdir()
                      if unicodedata.normalize("NFC", p.name) == target), None)
        if match is None:
            raise FileNotFoundError(f"경로를 찾을 수 없습니다: {current / part}")
        current = match
    return current

DATA_DIRS = {
    key: resolve_unicode_path(BASE_DIR / relative)
    for key, relative in {
        "train_source": "Training/01.원천데이터",
        "train_label": "Training/02.라벨링데이터",
        "val_source": "Validation/01.원천데이터",
        "val_label": "Validation/02.라벨링데이터",
    }.items()
}
for name, path in DATA_DIRS.items():
    print(name, path, "JSON 파일:", len(list(path.rglob("*.json"))))

train_source /Volumes/Samsung_T5/책_AI/013.동화 이해도 테스트를 위한 질의응답쌍 생성 데이터/3.개방데이터/1.데이터/Training/01.원천데이터 JSON 파일: 1446
train_label /Volumes/Samsung_T5/책_AI/013.동화 이해도 테스트를 위한 질의응답쌍 생성 데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터 JSON 파일: 1446
val_source /Volumes/Samsung_T5/책_AI/013.동화 이해도 테스트를 위한 질의응답쌍 생성 데이터/3.개방데이터/1.데이터/Validation/01.원천데이터 JSON 파일: 286
val_label /Volumes/Samsung_T5/책_AI/013.동화 이해도 테스트를 위한 질의응답쌍 생성 데이터/3.개방데이터/1.데이터/Validation/02.라벨링데이터 JSON 파일: 286


## 3. 문단별 다중 질문·정답 추출

In [4]:
def load_paragraph_samples(folder):
    rows = []
    bad_files = 0
    for path in sorted(folder.rglob("*.json")):
        try:
            with path.open("r", encoding="utf-8-sig") as f:
                data = json.load(f)
        except (OSError, json.JSONDecodeError) as e:
            bad_files += 1
            print("읽기 실패:", path.name, str(e)[:100])
            continue
        for paragraph in data.get("paragraphInfo", []):
            context = paragraph.get("srcText")
            if not isinstance(context, str) or not context.strip():
                continue
            pairs = []
            for qa in paragraph.get("queAnsPairInfo", []) or []:
                question = qa.get("question")
                if not isinstance(question, str) or not question.strip():
                    continue
                answers = []
                for i in range(1, 6):
                    a = qa.get(f"ansM{i}")
                    if isinstance(a, str) and a.strip() and a.strip() not in answers:
                        answers.append(a.strip())
                if answers:
                    pairs.append({"question": question.strip(), "answers": answers})
            if pairs:
                rows.append({"context": context.strip(), "qa_pairs": pairs})
    print(f"{folder.name}: {len(rows):,}문단, 읽기 실패 {bad_files}개")
    return rows

train_rows = load_paragraph_samples(DATA_DIRS["train_label"])
val_rows = load_paragraph_samples(DATA_DIRS["val_label"])
random.Random(SEED).shuffle(train_rows)
random.Random(SEED).shuffle(val_rows)
assert train_rows and val_rows, "학습 또는 검증 데이터가 비어 있습니다."
print("학습 예시:", train_rows[0])

02.라벨링데이터: 28,050문단, 읽기 실패 0개
02.라벨링데이터: 3,301문단, 읽기 실패 0개
학습 예시: {'context': '"그 때는 우리 집에 와서 지내면 되지 않겠는가. 돈도 없이 어떻게 탐험을 떠난단 말인가." "남이 읽어 주지도 않는 원고보다 탐험을 하는 것이 더 사람들의 흥미를 끌 것 같네."', 'qa_pairs': [{'question': '남이 읽어 주지도 않는 원고보다 무엇을 하는 것이 사람들의 흥미를 끌 것 같다고 말했나요?', 'answers': ['탐험하는 것이 사람들의 흥미를 더 끌 것 같다고 했어요.']}, {'question': '무엇도 없이 어떻게 탐험을 떠나냐고 말했나요?', 'answers': ['돈']}]}


## 4. 프롬프트 구성

In [5]:
def build_prompt(context):
    return f"""다음 동화 내용을 읽고, 내용에 근거한 서로 다른 질문과 정답을 여러 개 생성하세요.
질문과 정답은 한국어로 작성하고, 동화에 없는 사실은 만들어내지 마세요.

동화 내용:
{context}"""

def build_response(qa_pairs):
    blocks = []
    for idx, qa in enumerate(qa_pairs, start=1):
        blocks.append(f"질문 {idx}: {qa['question']}\n정답 {idx}: {', '.join(qa['answers'])}")
    return "\n\n".join(blocks)

print(build_prompt(train_rows[0]["context"]))
print("\n--- 정답 ---\n", build_response(train_rows[0]["qa_pairs"]))

다음 동화 내용을 읽고, 내용에 근거한 서로 다른 질문과 정답을 여러 개 생성하세요.
질문과 정답은 한국어로 작성하고, 동화에 없는 사실은 만들어내지 마세요.

동화 내용:
"그 때는 우리 집에 와서 지내면 되지 않겠는가. 돈도 없이 어떻게 탐험을 떠난단 말인가." "남이 읽어 주지도 않는 원고보다 탐험을 하는 것이 더 사람들의 흥미를 끌 것 같네."

--- 정답 ---
 질문 1: 남이 읽어 주지도 않는 원고보다 무엇을 하는 것이 사람들의 흥미를 끌 것 같다고 말했나요?
정답 1: 탐험하는 것이 사람들의 흥미를 더 끌 것 같다고 했어요.

질문 2: 무엇도 없이 어떻게 탐험을 떠나냐고 말했나요?
정답 2: 돈


## 5. 토크나이저와 LoRA 모델

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
# LoRA 적용 (한 번만 실행)
model = get_peft_model(model, lora_config)

# 메모리 최적화
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# MPS로 이동
model.to(DEVICE)

model.print_trainable_parameters()

Loading weights: 100%|██████████| 146/146 [00:01<00:00, 105.54it/s]


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


## 6. SFT 토큰화 및 길이 검사

In [7]:
def tokenize_sft(row):
    user_text = build_prompt(row["context"])
    assistant_text = build_response(row["qa_pairs"])
    user_messages = [{"role": "user", "content": user_text}]
    full_messages = user_messages + [{"role": "assistant", "content": assistant_text}]
    prompt_ids = tokenizer.apply_chat_template(
        user_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    )
    # 일부 transformers 버전은 list 대신 BatchEncoding을 반환할 수 있음
    if hasattr(prompt_ids, "get"):
        prompt_ids = prompt_ids["input_ids"]
    if hasattr(full_ids, "get"):
        full_ids = full_ids["input_ids"]
    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("채팅 템플릿의 프롬프트 접두사가 일치하지 않습니다.")
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    valid = len(full_ids) <= MAX_LENGTH and any(v != -100 for v in labels)
    return {
        "input_ids": full_ids if valid else [],
        "attention_mask": [1] * len(full_ids) if valid else [],
        "labels": labels if valid else [],
        "valid": valid,
        "token_length": len(full_ids),
    }

train_raw = Dataset.from_list(train_rows)
val_raw = Dataset.from_list(val_rows)
train_mapped = train_raw.map(tokenize_sft, remove_columns=train_raw.column_names)
val_mapped = val_raw.map(tokenize_sft, remove_columns=val_raw.column_names)

train_tokenized = train_mapped.filter(lambda x: x["valid"]).remove_columns(["valid", "token_length"])
val_tokenized = val_mapped.filter(lambda x: x["valid"]).remove_columns(["valid", "token_length"])

for name, before, after in [
    ("Training", train_mapped, train_tokenized),
    ("Validation", val_mapped, val_tokenized),
]:
    print(f"{name}: 전체 {len(before):,}, 사용 {len(after):,}, 제외 {len(before)-len(after):,}")
    print("최대 길이:", max(before["token_length"]))
assert len(train_tokenized) > 0 and len(val_tokenized) > 0, "사용 가능한 데이터가 없습니다."
print("예상 총 optimizer steps:", math.ceil(len(train_tokenized) / GRAD_ACCUM) * EPOCHS)

Filter: 100%|██████████| 3301/3301 [00:00<00:00, 70629.33 examples/s]

Training: 전체 28,050, 사용 27,998, 제외 52
최대 길이: 620
Validation: 전체 3,301, 사용 3,295, 제외 6
최대 길이: 570
예상 총 optimizer steps: 21000


## 7.동적 패딩과 Trainer 설정

In [8]:
@dataclass
class SFTCollator:
    pad_token_id: int

    def __call__(self, features):
        max_len = max(len(x["input_ids"]) for x in features)
        result = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in features:
            n = max_len - len(x["input_ids"])
            result["input_ids"].append(x["input_ids"] + [self.pad_token_id] * n)
            result["attention_mask"].append(x["attention_mask"] + [0] * n)
            result["labels"].append(x["labels"] + [-100] * n)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in result.items()}

collator = SFTCollator(tokenizer.pad_token_id)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    optim="adamw_torch",
    report_to="wandb",
    run_name=RUN_NAME,
    remove_unused_columns=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    torch_empty_cache_steps=20,
)

class MemoryMonitorCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.backends.mps.is_available():
            allocated = torch.mps.current_allocated_memory() / 1024**3
            driver = torch.mps.driver_allocated_memory() / 1024**3
            ram = psutil.virtual_memory()

            print(
                f"[Step {state.global_step}] "
                f"MPS allocated: {allocated:.2f} GiB | "
                f"MPS driver: {driver:.2f} GiB | "
                f"RAM used: {ram.percent:.1f}%"
            )

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=collator,
    callbacks=[MemoryMonitorCallback()],
)

print("체크포인트:", OUTPUT_DIR)

print("\n학습 설정")
print("Gradient Checkpointing:", args.gradient_checkpointing)
print("Batch Size:", args.per_device_train_batch_size)
print("Gradient Accumulation:", args.gradient_accumulation_steps)

print("\n데이터셋")
print("학습 샘플:", len(train_tokenized))
print("검증 샘플:", len(val_tokenized))

print("\n토큰 길이")
print(
    "학습 최대 토큰 길이:",
    max(len(x["input_ids"]) for x in train_tokenized)
)

print(
    "검증 최대 토큰 길이:",
    max(len(x["input_ids"]) for x in val_tokenized)
)

print("\n모델 설정")
print("Use Cache:", model.config.use_cache)
print(
    "Gradient Checkpointing 활성화:",
    model.is_gradient_checkpointing
)

체크포인트: /Users/dongho/InterBook/models/multi_qa_wandb_1000steps_mps32gb/checkpoints

학습 설정
Gradient Checkpointing: True
Batch Size: 1
Gradient Accumulation: 4

데이터셋
학습 샘플: 27998
검증 샘플: 3295

토큰 길이
학습 최대 토큰 길이: 512
검증 최대 토큰 길이: 507

모델 설정
Use Cache: False
Gradient Checkpointing 활성화: True


## 8.학습 실행 및 LoRA 어댑터 저장

In [9]:
# 새 학습
trainer.train()

# 학습 완료 후 어댑터 및 토크나이저 저장
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
wandb.finish()
print("LoRA 어댑터 저장 완료:", ADAPTER_DIR)

wandb: WARNING Path / wasn't read/writable
wandb: WARNING Falling back to temporary directory /var/folders/07/9vtbg4yd76q033v0h_l9b21c0000gn/T.


Step,Training Loss,Validation Loss
1000,0.960395,0.971395
2000,0.919569,0.924260
3000,0.905912,0.909963
4000,0.859774,0.874687
5000,0.884060,0.873091
6000,0.844786,0.847737
7000,0.843396,0.849877
8000,0.804230,0.844514
9000,0.790344,0.827767
10000,0.811178,0.822808


[Step 20] MPS allocated: 2.32 GiB | MPS driver: 14.06 GiB | RAM used: 67.8%
[Step 40] MPS allocated: 2.32 GiB | MPS driver: 12.62 GiB | RAM used: 64.2%
[Step 60] MPS allocated: 2.32 GiB | MPS driver: 11.89 GiB | RAM used: 63.7%
[Step 80] MPS allocated: 2.32 GiB | MPS driver: 14.11 GiB | RAM used: 67.2%
[Step 100] MPS allocated: 2.32 GiB | MPS driver: 14.31 GiB | RAM used: 66.9%
[Step 120] MPS allocated: 2.32 GiB | MPS driver: 11.52 GiB | RAM used: 62.4%
[Step 140] MPS allocated: 2.32 GiB | MPS driver: 12.70 GiB | RAM used: 64.3%
[Step 160] MPS allocated: 2.32 GiB | MPS driver: 14.11 GiB | RAM used: 67.6%
[Step 180] MPS allocated: 2.32 GiB | MPS driver: 13.75 GiB | RAM used: 67.3%
[Step 200] MPS allocated: 2.32 GiB | MPS driver: 12.98 GiB | RAM used: 66.2%
[Step 220] MPS allocated: 2.32 GiB | MPS driver: 11.48 GiB | RAM used: 62.0%
[Step 240] MPS allocated: 2.32 GiB | MPS driver: 12.01 GiB | RAM used: 63.1%
[Step 260] MPS allocated: 2.32 GiB | MPS driver: 13.74 GiB | RAM used: 67.6%
[St

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/loss,█▆▅▄▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
eval/runtime,██▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▂▂▂▂
eval/samples_per_second,▁▁▇▇▇▇▇▇▇█▇▇█████▇▇▇▇
eval/steps_per_second,▁▁▇▇▇▇▇▇▇█▇▇█████▇▇▇▇
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▆▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,▃█▅▃▂▃▄▄▅▄▄▃▄▆▂▅▃▃▂▇▆▅▁▅▄▃▂▄▃▃▄▃▅▅▃▄▃▄▄▄
train/learning_rate,███████████▇▇▇▇▆▅▅▅▅▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train/loss,▇█▇▇▇▄▆▄▄▇▅▃▄▄▄▃▄▄▃▂▃▅▃▂▁▂▄▃▃▄▁▃▂▃▂▁▂▁▁▂
eval/loss,0.80406
eval/runtime,898.0952


LoRA 어댑터 저장 완료: /Users/dongho/InterBook/models/multi_qa_wandb_1000steps_mps32gb/adapter


## 9. 체크포인트에서 재개 (필요할 때만)

In [10]:
# 필요할 때만 주석을 풀고 실행
# checkpoints = sorted(Path(OUTPUT_DIR).glob("checkpoint-*"),
#                      key=lambda p: int(p.name.split("-")[-1]))
# if not checkpoints:
#     raise FileNotFoundError("저장된 체크포인트가 없습니다.")
# print("재개:", checkpoints[-1])
# trainer.train(resume_from_checkpoint=str(checkpoints[-1]))
# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)
# wandb.finish()

## 10. 동화 내용만으로 여러 질문·정답 생성

In [11]:
# 테스트용 동화 내용
test_context = """할머니는 꽃밭에 물을 주고 있었어요.
갑자기 하늘에서 우박이 떨어지자 할머니는 집 안으로 뛰어 들어갔어요."""

# 추론 모드
model.eval()
model.config.use_cache = True

# 프롬프트 구성
messages = [
    {
        "role": "user",
        "content": build_prompt(test_context)
    }
]

# 토큰화
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(DEVICE)

# 질문·정답 생성
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

# 생성 결과만 추출
result_ids = generated_ids[
    0,
    inputs["input_ids"].shape[1]:
]

print(tokenizer.decode(
    result_ids,
    skip_special_tokens=True
))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


질문 1: 할머니는 어디에 물을 주고 있었나요?
정답 1: 꽃밭

질문 2: 할머니는 어디로 뛰어 들어갔나요?
정답 2: 집 안

질문 3: 할머니는 어디에서 우박이 떨어졌나요?
정답 3: 하늘

질문 4: 할머니는 어디로 뛰어 들어갔나요?
정답 4: 집 안으로 뛰어 들어갔어요.

질문 5: 할머니는 어디에 물을 주고 있었나요?
정답 5: 꽃밭


## 11. 학습 로그 확인

In [12]:
for entry in trainer.state.log_history:
    print(entry)

{'loss': 1.3411211967468262, 'grad_norm': 1.7394126653671265, 'learning_rate': 9.999995463720243e-05, 'epoch': 0.002857346953353811, 'step': 20}
{'loss': 1.1105629920959472, 'grad_norm': 2.5313820838928223, 'learning_rate': 9.999952901162184e-05, 'epoch': 0.005714693906707622, 'step': 40}
{'loss': 1.1498053550720215, 'grad_norm': 2.7717831134796143, 'learning_rate': 9.999865536289802e-05, 'epoch': 0.008572040860061433, 'step': 60}
{'loss': 1.2176395416259767, 'grad_norm': 2.761892318725586, 'learning_rate': 9.999733369885937e-05, 'epoch': 0.011429387813415244, 'step': 80}
{'loss': 1.1660767555236817, 'grad_norm': 3.6052651405334473, 'learning_rate': 9.999556403134869e-05, 'epoch': 0.014286734766769055, 'step': 100}
{'loss': 1.099937915802002, 'grad_norm': 2.8949546813964844, 'learning_rate': 9.99933463762232e-05, 'epoch': 0.017144081720122866, 'step': 120}
{'loss': 1.1177434921264648, 'grad_norm': 2.454465627670288, 'learning_rate': 9.999068075335429e-05, 'epoch': 0.020001428673476675,

In [15]:
with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=[
            tokenizer.eos_token_id,
            tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ],
    )

result_ids = generated_ids[0, inputs["input_ids"].shape[1]:]

print("생성 토큰 수:", len(result_ids))
print("마지막 토큰:", tokenizer.convert_ids_to_tokens(result_ids[-1].item()))

print("\n--- 생성 결과 ---")
print(tokenizer.decode(result_ids, skip_special_tokens=True))

생성 토큰 수: 90
마지막 토큰: <|eot_id|>

--- 생성 결과 ---
질문 1: 할머니는 어디에 물을 주고 있었나요?
정답 1: 꽃밭

질문 2: 할머니는 어디로 뛰어 들어갔나요?
정답 2: 집 안

질문 3: 하늘에서 우박이 떨어지자 할머니는 어떤 행동을 했나요?
정답 3: 집 안으로 뛰어 들어갔어요.


In [16]:
model.eval()

for i in range(5):
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[
                tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids("<|eot_id|>")
            ],
        )

    result_ids = generated_ids[0, inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        result_ids,
        skip_special_tokens=True
    )

    print(f"\n{'=' * 20} 테스트 {i+1} {'=' * 20}")
    print(result)


==================== 테스트 1 ====================
질문 1: 하늘에서 떨어진 우박을 본 할머니는 어떻게 했나요?
정답 1: 집 안으로 뛰어 들어갔어요.

질문 2: 꽃밭에 물을 주고 있던 사람은 누구인가요?
정답 2: 할머니

==================== 테스트 2 ====================
질문 1: 꽃밭에 물을 주고 있는 사람은 누구인가요?
정답 1: 할머니

질문 2: 할머니는 어디로 뛰어 들어갔나요?
정답 2: 집 안

질문 3: 할머니가 집 안으로 뛰어 들어간 이유는 무엇인가요?
정답 3: 갑자기 우박이 떨어지자요.

==================== 테스트 3 ====================
질문 1: 꽃밭에 물을 주고 있는 사람은 누구인가요?
정답 1: 할머니

질문 2: 할머니가 하늘에서 우박이 떨어지자 어떤 행동을 했나요?
정답 2: 할머니는 집 안으로 뛰어 들어갔어요.

==================== 테스트 4 ====================
질문 1: 할머니는 어디에 물을 주고 있었나요?
정답 1: 꽃밭

질문 2: 하늘에서 우박이 떨어지자 할머니는 어떻게 했나요?
정답 2: 집 안으로 뛰어 들어갔어요.

==================== 테스트 5 ====================
질문 1: 꽃밭에 물을 주고 있는 사람은 누구인가요?
정답 1: 할머니

질문 2: 할머니는 어디로 뛰어 들어갔나요?
정답 2: 집 안

질문 3: 할머니가 집 안으로 뛰어 들어간 이유는 무엇일까요?
정답 3: 갑자기 우박이 떨어졌기 때문이에요.
